# Лаборатория «Интуиция ML» — Ансамбли

## Деревья, лес, бэггинг, бустинг, стекинг — что они физически делают

> **Это финальный ноутбук лабораторной.** Рекомендуемый порядок чтения: `02_regression_intuition` → `01_classification_intuition` → **этот ноутбук**.
> К этому моменту ты уже понимаешь, как работают линейные модели через градиентный спуск. Здесь рассматриваем **нелинейные** модели — деревья и их ансамбли — на тех же двух датасетах, что и раньше.

В предыдущих ноутбуках мы работали с **линейными** моделями — разделяющая граница (или прогноз) — это прямая/плоскость. Теперь рассмотрим модели, которые могут строить **ступенчатые / нелинейные** границы.

Используем **те же два датасета**:
- BMI (рост, вес → толстый/худой) для классификации;
- квартиры (площадь, км → цена) для регрессии.

Раз признаков всего два — мы можем **в плоскости** нарисовать decision boundary каждой модели и увидеть, что она делает.

## 0. Импорты и данные

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree
from sklearn.ensemble import (
    RandomForestClassifier, RandomForestRegressor,
    BaggingClassifier, BaggingRegressor,
    GradientBoostingClassifier, GradientBoostingRegressor,
    StackingClassifier, StackingRegressor,
)
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, mean_squared_error
from sklearn.preprocessing import StandardScaler

from data_generators import make_bmi_dataset, make_apartment_dataset
from plotting import plot_decision_boundary, plot_regression_surface

plt.rcParams['figure.dpi'] = 100
sns.set_style("whitegrid")
np.random.seed(42)

# Данные
df_cls = make_bmi_dataset(n=200, label_noise=0.05, random_state=42)
X_cls = df_cls[['height', 'weight']].to_numpy()
y_cls = df_cls['is_fat'].to_numpy()
X_cls_train, X_cls_test, y_cls_train, y_cls_test = train_test_split(
    X_cls, y_cls, test_size=0.3, random_state=42, stratify=y_cls
)

df_reg = make_apartment_dataset(n=200, noise_scale=5000, random_state=42)
X_reg = df_reg[['area', 'dist']].to_numpy()
y_reg = df_reg['price'].to_numpy()
X_reg_train, X_reg_test, y_reg_train, y_reg_test = train_test_split(
    X_reg, y_reg, test_size=0.3, random_state=42
)

print(f'Классификация: train {X_cls_train.shape}, test {X_cls_test.shape}')
print(f'Регрессия:     train {X_reg_train.shape}, test {X_reg_test.shape}')

## Часть 1. Решающее дерево

### Идея

В отличие от регрессий, дерево **не имеет градиента** и не использует loss напрямую. Оно жадно делает разбиения вида «если feature_j > threshold, иди влево, иначе вправо», выбирая разбиение, которое максимально уменьшает **примесь** (Gini для классификации, MSE для регрессии) в дочерних узлах.

Получается ступенчатая граница — строго горизонтальные/вертикальные линии в исходных признаках.

### Дерево на классификации (BMI)

In [ ]:
dt_cls = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_cls_train, y_cls_train)
print(f'accuracy на test: {dt_cls.score(X_cls_test, y_cls_test):.3f}')

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
plot_tree(dt_cls, ax=axes[0], feature_names=['height', 'weight'],
          class_names=['худой', 'толстый'], filled=True, rounded=True)
axes[0].set_title('Структура дерева (max_depth=3)')

plot_decision_boundary(dt_cls.predict, X_cls, y_cls, ax=axes[1],
                       title='Decision boundary — характерные ступеньки',
                       feature_names=('рост', 'вес'))
plt.tight_layout()
plt.show()

**Видно:** граница состоит **только из горизонтальных и вертикальных отрезков**. Это и есть «ступенчатая» природа дерева — каждое разбиение режет плоскость по одному признаку.

### Эффект max_depth — переобучение глазами

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, depth in zip(axes, [1, 3, 15]):
    m = DecisionTreeClassifier(max_depth=depth, random_state=42).fit(X_cls_train, y_cls_train)
    train_acc = m.score(X_cls_train, y_cls_train)
    test_acc = m.score(X_cls_test, y_cls_test)
    plot_decision_boundary(m.predict, X_cls, y_cls, ax=ax,
                           title=f'max_depth={depth}\ntrain={train_acc:.2f}, test={test_acc:.2f}',
                           feature_names=('рост', 'вес'))
plt.tight_layout()
plt.show()

- **depth=1** — слишком грубо (одна линия), низкая точность и на train, и на test (**недообучение**);
- **depth=3** — хороший компромисс;
- **depth=15** — граница «пиксельная», train близок к 100%, но **test хуже** — это **переобучение**: дерево заучило шум.

### Дерево на регрессии (квартиры)

In [ ]:
dt_reg = DecisionTreeRegressor(max_depth=4, random_state=42).fit(X_reg_train, y_reg_train)
mse = mean_squared_error(y_reg_test, dt_reg.predict(X_reg_test))
print(f'MSE на test: {mse:.0f}')

fig, ax = plt.subplots(figsize=(14, 6))
plot_tree(dt_reg, ax=ax, feature_names=['area', 'dist'], filled=True, rounded=True, fontsize=9)
ax.set_title('Дерево регрессии для цены (max_depth=4)')
plt.show()

In [ ]:
fig = plot_regression_surface(dt_reg.predict, X_reg, y_reg,
                              feature_names=('area', 'dist'),
                              target_name='price',
                              title='Поверхность предсказаний дерева — характерные ступеньки')
fig.show()

Это и есть ключевой признак дерева: предсказание — **кусочно-постоянная функция**. В пределах каждого «листа» дерево выдаёт одно и то же значение, граница между листьями — резкая ступенька.

## Часть 2. Случайный лес

### Идея

Вместо одного дерева обучим **много** (например 100), каждое — на:
- **bootstrap-выборке** (сэмпл с возвращением исходного размера);
- **случайном подмножестве признаков** при каждом split'е (этим RF отличается от чистого bagging'а).

Финальное предсказание — среднее (регрессия) или голосование (классификация). Поскольку отдельные деревья **разные** (видели разные данные/признаки), их ошибки **не коррелируют** — усреднение их сильно сглаживает.

### Сравнение boundary'ев: одно дерево vs 5 разных деревьев vs весь лес

In [ ]:
rf_cls = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42, n_jobs=-1).fit(X_cls_train, y_cls_train)
print(f'RF accuracy на test: {rf_cls.score(X_cls_test, y_cls_test):.3f}')
print(f'Сравни: одно дерево depth=5: {DecisionTreeClassifier(max_depth=5, random_state=42).fit(X_cls_train, y_cls_train).score(X_cls_test, y_cls_test):.3f}')

# Берём первые 5 деревьев из леса и рисуем их boundary отдельно
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
for i, ax in enumerate(axes.flat[:5]):
    tree = rf_cls.estimators_[i]
    plot_decision_boundary(tree.predict, X_cls, y_cls, ax=ax,
                           title=f'Дерево #{i+1} из леса (видело свой bootstrap)',
                           feature_names=('рост', 'вес'))

plot_decision_boundary(rf_cls.predict, X_cls, y_cls, ax=axes.flat[5],
                       title=f'Весь лес (100 деревьев) — усреднение',
                       feature_names=('рост', 'вес'))
plt.tight_layout()
plt.show()

**Заметь:** отдельные деревья **РАЗНЫЕ** — одни режут плоскость одним способом, другие иначе. Каждое в одиночку ошибается, но **усреднение убирает шум** и оставляет общий правильный паттерн. Граница леса сильно более **гладкая** и устойчивая.

### То же для регрессии

In [ ]:
rf_reg = RandomForestRegressor(n_estimators=100, max_depth=6, random_state=42, n_jobs=-1).fit(X_reg_train, y_reg_train)
print(f'RF MSE на test: {mean_squared_error(y_reg_test, rf_reg.predict(X_reg_test)):.0f}')
print(f'Одно дерево MSE на test: {mean_squared_error(y_reg_test, dt_reg.predict(X_reg_test)):.0f}')

fig = plot_regression_surface(rf_reg.predict, X_reg, y_reg,
                              feature_names=('area', 'dist'),
                              target_name='price',
                              title='Лес из 100 деревьев — поверхность сильно сглажена')
fig.show()

Сравни с поверхностью одного дерева выше: там были резкие плато, здесь — почти плавная поверхность, близкая к **истинной плоскости** (`price ≈ 1500·area - 800·dist + 50000`). Усреднение многих ступенчатых функций → почти гладкая.

## Часть 3. Бэггинг

### Идея

**Bagging = Bootstrap AGGregating**. По сути это RF без второй случайности (без сэмплирования признаков). Каждое дерево учится на bootstrap-выборке, потом усреднение.

Покажем bootstrap явно:

In [ ]:
n = len(X_cls_train)
bootstrap_idx = np.random.choice(n, size=n, replace=True)
unique_in = np.unique(bootstrap_idx)
oob_idx = np.setdiff1d(np.arange(n), unique_in)

print(f'Размер обучающей выборки: {n}')
print(f'Размер bootstrap (с возвращением): {n} (но уникальных всего {len(unique_in)})')
print(f'Out-of-bag (не попали в bootstrap): {len(oob_idx)} ({100*len(oob_idx)/n:.0f}%)')
print('\nТипично OOB ~ 37% — потому что вероятность не попасть = (1-1/n)^n → 1/e ≈ 0.37')

In [ ]:
bag_cls = BaggingClassifier(estimator=DecisionTreeClassifier(max_depth=5),
                            n_estimators=100, random_state=42, n_jobs=-1).fit(X_cls_train, y_cls_train)
print(f'Bagging accuracy на test: {bag_cls.score(X_cls_test, y_cls_test):.3f}')

# дисперсия одиночного дерева vs bagging — запустим 10 раз с разными seed
single_scores = []
bag_scores = []
for s in range(10):
    Xt, _, yt, _ = train_test_split(X_cls, y_cls, test_size=0.3, random_state=s, stratify=y_cls)
    single_scores.append(DecisionTreeClassifier(max_depth=5, random_state=s).fit(Xt, yt).score(X_cls_test, y_cls_test))
    bag_scores.append(BaggingClassifier(estimator=DecisionTreeClassifier(max_depth=5),
                                        n_estimators=50, random_state=s, n_jobs=-1).fit(Xt, yt).score(X_cls_test, y_cls_test))

fig, ax = plt.subplots(figsize=(8, 5))
ax.boxplot([single_scores, bag_scores], labels=['одно дерево\n(depth=5)', 'bagging\n(50 деревьев)'])
ax.set_ylabel('accuracy на test')
ax.set_title('Дисперсия точности по 10 разным train-выборкам:\nbagging стабильнее')
plt.show()

print(f'std одного дерева: {np.std(single_scores):.4f}')
print(f'std bagging:       {np.std(bag_scores):.4f}')

**Точечная мысль:** дисперсия точности у bagging **меньше**. Усреднение многих моделей не только улучшает точность, но и делает её **более стабильной**.

## Часть 4. Бустинг

### Идея — самая важная для интуиции

В bagging'е и RF деревья **независимы** (учатся параллельно). В **бустинге** деревья **зависимы**: каждое следующее обучается **на ошибках предыдущего ансамбля**.

Покажем это руками на регрессии за 3 итерации.

In [ ]:
from sklearn.tree import DecisionTreeRegressor

X_train, y_train = X_reg_train, y_reg_train
lr = 0.3  # learning rate бустинга — не путать с lr градиентного спуска

# Итерация 0: предсказываем средним (база)
y_pred_total = np.full_like(y_train, y_train.mean(), dtype=float)
residuals_history = [y_train - y_pred_total]

# 3 итерации
trees = []
for i in range(3):
    residuals = y_train - y_pred_total
    # дерево учится предсказывать ОСТАТКИ, а не саму цену
    tree = DecisionTreeRegressor(max_depth=3, random_state=42).fit(X_train, residuals)
    pred_resid = tree.predict(X_train)
    # обновляем общий прогноз: добавляем lr * предсказание остатков
    y_pred_total = y_pred_total + lr * pred_resid
    trees.append(tree)
    residuals_history.append(y_train - y_pred_total)

print('MSE после каждой итерации бустинга:')
for i, r in enumerate(residuals_history):
    print(f'  итерация {i}: MSE = {(r**2).mean():.0f},  std остатков = {r.std():.0f}')

In [ ]:
# Визуализация: распределение остатков на каждой итерации
fig, axes = plt.subplots(1, 4, figsize=(20, 4), sharex=True, sharey=True)
for i, (ax, r) in enumerate(zip(axes, residuals_history)):
    ax.hist(r, bins=30, edgecolor='black', alpha=0.7)
    ax.axvline(0, color='red', linestyle='--')
    ax.set_title(f'остатки после {i} деревьев\nstd = {r.std():.0f}')
    ax.set_xlabel('y - ŷ')
plt.suptitle('Бустинг: остатки прижимаются к нулю с каждой итерацией')
plt.tight_layout()
plt.show()

Видно: **дисперсия остатков уменьшается** с каждым деревом. Это и есть бустинг — последовательное «закрашивание» того, что предыдущий ансамбль не уловил.

Формула обновления:
$$
\hat{y}^{(t+1)} = \hat{y}^{(t)} + \text{lr} \cdot h_t(x),
$$
где $h_t$ — новое дерево, обученное на остатках $y - \hat{y}^{(t)}$.

### Теперь sklearn'овский GradientBoosting (классификация и регрессия)

In [ ]:
gb_cls = GradientBoostingClassifier(n_estimators=200, max_depth=3, learning_rate=0.1, random_state=42).fit(X_cls_train, y_cls_train)
gb_reg = GradientBoostingRegressor(n_estimators=200, max_depth=3, learning_rate=0.1, random_state=42).fit(X_reg_train, y_reg_train)
print(f'GB classification accuracy: {gb_cls.score(X_cls_test, y_cls_test):.3f}')
print(f'GB regression MSE:          {mean_squared_error(y_reg_test, gb_reg.predict(X_reg_test)):.0f}')

fig, ax = plt.subplots(figsize=(7, 6))
plot_decision_boundary(gb_cls.predict, X_cls, y_cls, ax=ax,
                       title='GradientBoosting — гладкая, но не круговая граница',
                       feature_names=('рост', 'вес'))
plt.show()

### Влияние learning_rate в бустинге

In [ ]:
lrs = [0.01, 0.1, 1.0]
fig, ax = plt.subplots(figsize=(8, 4))
for lr in lrs:
    m = GradientBoostingRegressor(n_estimators=200, max_depth=3, learning_rate=lr, random_state=42)
    m.fit(X_reg_train, y_reg_train)
    # train MSE по итерациям через staged_predict
    train_losses = [mean_squared_error(y_reg_train, p) for p in m.staged_predict(X_reg_train)]
    ax.plot(train_losses, label=f'lr={lr}', linewidth=2)
ax.set_xlabel('число деревьев')
ax.set_ylabel('train MSE (log)')
ax.set_yscale('log')
ax.set_title('Бустинг: learning_rate балансирует скорость vs стабильность')
ax.legend()
plt.show()

- **lr=0.01:** медленно, нужно много деревьев;
- **lr=0.1:** хороший дефолт;
- **lr=1.0:** слишком агрессивно, можем переобучиться или скакать.

## Часть 5. Стекинг

### Идея

В отличие от bagging/boosting, в стекинге используются **разнотипные** базовые модели (например, дерево + linear + KNN). Каждая делает своё предсказание, а **мета-модель** (обычно простой LogReg/LinReg) учится их **взвешивать**.

Это полезно когда разные модели сильны в разных «уголках» пространства признаков — мета-модель учится «доверять» им селективно.

In [ ]:
# Базовые модели — разные по природе
base_estimators = [
    ('logreg', LogisticRegression(max_iter=1000)),
    ('tree', DecisionTreeClassifier(max_depth=5, random_state=42)),
    ('knn', KNeighborsClassifier(n_neighbors=7)),
]

# Скейлинг для logreg и knn (для дерева неважно, но не повредит)
scaler = StandardScaler().fit(X_cls_train)
X_cls_train_s = scaler.transform(X_cls_train)
X_cls_test_s = scaler.transform(X_cls_test)

# Стекинг с logreg как мета-моделью
stack = StackingClassifier(estimators=base_estimators,
                           final_estimator=LogisticRegression(),
                           cv=5).fit(X_cls_train_s, y_cls_train)

print(f'Stacking accuracy: {stack.score(X_cls_test_s, y_cls_test):.3f}')

# базовые модели по отдельности
for name, est in base_estimators:
    est.fit(X_cls_train_s, y_cls_train)
    print(f'  {name}: {est.score(X_cls_test_s, y_cls_test):.3f}')

### Что такое «веса мета-модели» и как их интерпретировать

`final_estimator` — это `LogisticRegression`, и у него есть `.coef_`. Это коэффициенты ПРИ ПРЕДСКАЗАНИЯХ базовых моделей. То есть мета-модель буквально учится формуле «итоговое предсказание = $\alpha_1 \cdot \text{logreg} + \alpha_2 \cdot \text{tree} + \alpha_3 \cdot \text{knn}$».

In [ ]:
meta_coefs = stack.final_estimator_.coef_[0]
names = [n for n, _ in base_estimators]

print('Коэффициенты мета-модели (logreg как мета):')
for n, c in zip(names, meta_coefs):
    print(f'  {n}: {c:.3f}')
print(f'\nintercept мета-модели: {stack.final_estimator_.intercept_[0]:.3f}')
print()
print('Бóльший по модулю коэффициент → ансамбль больше доверяет этой базовой модели.')
print('(Знак тут немного условный, потому что мета-LogReg оперирует логитами.)')

fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(names, np.abs(meta_coefs), color=['C0', 'C1', 'C2'])
ax.set_xlabel('|коэффициент мета-модели|')
ax.set_title('Какой базовой модели стекинг доверяет больше')
plt.tight_layout()
plt.show()

## Часть 6. Финальное сравнение всех моделей

Свёдём в таблицу: для каждого датасета — точность всех 7 моделей.

In [ ]:
# Классификация
cls_models = {
    'LogReg': LogisticRegression(max_iter=1000),
    'DecisionTree(d=5)': DecisionTreeClassifier(max_depth=5, random_state=42),
    'RandomForest': RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42, n_jobs=-1),
    'Bagging': BaggingClassifier(estimator=DecisionTreeClassifier(max_depth=5),
                                 n_estimators=100, random_state=42, n_jobs=-1),
    'GradientBoosting': GradientBoostingClassifier(n_estimators=200, max_depth=3, random_state=42),
    'Stacking': stack,
}
cls_results = []
for name, m in cls_models.items():
    if name != 'Stacking':
        m.fit(X_cls_train_s, y_cls_train)
    score = m.score(X_cls_test_s, y_cls_test)
    cls_results.append((name, score))

print('Классификация (BMI → толстый/худой):')
for name, score in cls_results:
    print(f'  {name:25s} accuracy = {score:.3f}')

In [ ]:
# Регрессия
reg_models = {
    'LinReg': LinearRegression(),
    'DecisionTree(d=6)': DecisionTreeRegressor(max_depth=6, random_state=42),
    'RandomForest': RandomForestRegressor(n_estimators=100, max_depth=6, random_state=42, n_jobs=-1),
    'Bagging': BaggingRegressor(estimator=DecisionTreeRegressor(max_depth=6),
                                n_estimators=100, random_state=42, n_jobs=-1),
    'GradientBoosting': GradientBoostingRegressor(n_estimators=200, max_depth=3, random_state=42),
    'Stacking': StackingRegressor(estimators=[
        ('lin', LinearRegression()),
        ('tree', DecisionTreeRegressor(max_depth=6, random_state=42)),
        ('knn', KNeighborsRegressor(n_neighbors=7)),
    ], final_estimator=LinearRegression(), cv=5),
}
reg_results = []
for name, m in reg_models.items():
    m.fit(X_reg_train, y_reg_train)
    mse = mean_squared_error(y_reg_test, m.predict(X_reg_test))
    reg_results.append((name, mse))

print('\nРегрессия (квартиры):')
for name, mse in reg_results:
    print(f'  {name:25s} MSE = {mse:>12.0f}, RMSE = {np.sqrt(mse):>8.0f}')

### Decision boundary всех классификаторов в одной сетке

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
for ax, (name, m) in zip(axes.flat, cls_models.items()):
    plot_decision_boundary(m.predict, X_cls, y_cls, ax=ax,
                           title=name, feature_names=('рост', 'вес'))
plt.tight_layout()
plt.show()

**Сравни границы:**
- **LogReg** — прямая линия (единственный «истинно линейный» из всех).
- **DecisionTree** — резкие ступеньки.
- **RandomForest / Bagging / GradientBoosting** — плавные, потому что усреднение многих ступенчатых.
- **Stacking** — комбинация, форма зависит от того, какой базе мета-модель доверяет больше.

Для нашей задачи (BMI разделим примерно прямой) **LogReg** уже решает почти оптимально. На более сложных задачах с нелинейной границей выигрыш ансамблей был бы драматичнее.

## Часть 7. Что значит «интерпретируемость» для каждой модели

| Модель | Что можно сказать про предсказание |
|---|---|
| **LinReg / LogReg** | Коэффициент $w_j$ = «насколько изменится предсказание при +1 в признаке $j$». Самая интерпретируемая. |
| **Decision Tree** | Путь от корня к листу = последовательность правил «если height > 175 И weight > 80 → толстый». Можно прочитать вслух. |
| **Random Forest** | Индивидуальные деревья непрозрачны (их 100), но `feature_importances_` показывает, какие признаки используются чаще. |
| **Bagging** | То же что RF, без сэмплирования признаков. |
| **Gradient Boosting** | `feature_importances_` есть; для конкретного предсказания — SHAP/partial dependence (тяжёлая артиллерия, упоминаем). |
| **Stacking** | Коэффициенты мета-модели = «степень доверия» к каждому базовому алгоритму. |

In [ ]:
# feature importances у RF и GB
print('feature_importances_ у Random Forest:')
for f, imp in zip(['height', 'weight'], rf_cls.feature_importances_):
    print(f'  {f}: {imp:.3f}')
print()
print('feature_importances_ у GradientBoosting:')
for f, imp in zip(['height', 'weight'], gb_cls.feature_importances_):
    print(f'  {f}: {imp:.3f}')
print()
print('Видим: в обоих случаях вес (weight) важнее роста — что соответствует BMI = вес/рост².')

### Главный trade-off

Чем сложнее модель — тем выше потенциальная точность на сложных задачах, но **сложнее объяснить КОНКРЕТНОЕ предсказание**.

| Модель | Точность | Интерпретируемость одного предсказания |
|---|---|---|
| LinReg / LogReg | ★★ | ★★★★★ |
| DecisionTree | ★★★ | ★★★★ |
| RandomForest | ★★★★ | ★★ |
| Bagging | ★★★★ | ★★ |
| GradientBoosting | ★★★★★ | ★★ |
| Stacking | ★★★★★ | ★ |

Когда предсказание модели идёт человеку (банк, медицина, юр-сфера) — простота важнее точности. Когда важна точность ради точности (рекомендации, рекламные клики) — сложные модели.

## Чеклист «понял ли я»

1. Чем отличается random forest от bagging-а деревьев? *(RF дополнительно случайно сэмплирует ПРИЗНАКИ при каждом split'е)*
2. Какую функцию ошибки оптимизирует дерево? *(impurity: Gini для классификации, MSE для регрессии — жадно, без градиента)*
3. На чём учится N-ое дерево в boosting'е? *(на остатках $y - \hat{y}^{(N-1)}$ предыдущего ансамбля)*
4. Почему bagging уменьшает дисперсию? *(усреднение независимых ошибок: $\text{Var}(\bar{X}) = \sigma^2/n$)*
5. Что делает мета-модель в стекинге? *(учится взвешивать предсказания базовых моделей)*
6. Почему deep tree оверфитит, а deep RF — гораздо меньше? *(деревья в RF учились на разных bootstrap-выборках и разных подмножествах признаков, их шум некоррелирован и в среднем гасится)*
7. У какой модели decision boundary прямая? *(LogReg/LinReg — единственная истинно линейная среди наших)*

---

## Итог лабораторной

Ты прошёл путь от градиентного спуска по log-loss / MSE до сложных ансамблей. Главные выводы:

1. **Все «магии» — следствия цепного правила.** $\partial \mathcal{L}/\partial w_j \propto x_j$ — поэтому веса при больших признаках обновляются сильнее, поэтому скейлинг важен.
2. **Loss-поверхность визуализируема** для 2D. Минимум там, где $\nabla \mathcal{L} = 0$. GD просто скатывается туда — никакой магии.
3. **Скейлинг превращает овраг в круг**, и GD идёт прямо в минимум вместо зигзага.
4. **Дерево** — кусочно-постоянная функция со ступенчатой границей.
5. **Лес/бэггинг** — усреднение многих разных деревьев → гладкая граница, меньше дисперсия.
6. **Бустинг** — последовательное закрашивание остатков. Каждое следующее дерево учится на том, что осталось недосказанным.
7. **Стекинг** — мета-модель взвешивает разнотипные базовые модели.

Если все 22 вопроса самопроверки (8 + 7 + 7 в трёх ноутбуках) разложились по полочкам — материал освоен на интуитивном уровне, а не вызубрен. Удачи в дальнейших лабах!